# STAT3009 — Midterm workflow rehearsal

**Suggested time:** 35–40 minutes

This is a rehearsal, not a copy of the exam. You will build a small pipeline,
write reusable helpers, fix the model workflow, and compare local scores. The
test labels are visible here only for practice; the real midterm hides them and
uses `submit()`.

**Goal:** read data → encode IDs → baselines → MF → 4-fold CV → compare.

## 1. Check the data contract (3 min)

`raw_ratings.csv` has `user, item, rating`; `test_pairs.csv` has `user, item`;
`test_labels.csv` contains local test ratings.

Look at the tables and answer aloud:

1. How many observed ratings and test pairs are there?
2. Why can `U71371` not be used directly as a NumPy array index?
3. What must be true about the user and item vocabularies before fitting?

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import LabelEncoder

RANDOM_STATE = 3009
for candidate in [Path('.'), Path('rehearsal'), Path('nb/rehearsal')]:
    if (candidate / 'raw_ratings.csv').exists():
        DATA_DIR = candidate
        break
else:
    raise FileNotFoundError('Open this notebook from the rehearsal folder.')
sys.path.insert(0, str(DATA_DIR.resolve()))

from TabRS import GlobalMeanRS, UserMeanRS, ItemMeanRS, BiasedMF

np.random.seed(RANDOM_STATE)

train = pd.read_csv(DATA_DIR / 'raw_ratings.csv')
test = pd.read_csv(DATA_DIR / 'test_pairs.csv')
test_labels = pd.read_csv(DATA_DIR / 'test_labels.csv')['rating'].to_numpy(dtype=float)

print('Observed ratings:', len(train))
print('Test pairs:', len(test))
display(train.head())
display(test.head())

## 2. Package the preprocessing (8 min)

Write **one function** that returns the arrays and vocabulary sizes:

```python
X_train, X_test, y_train, n_users, n_items = encode_ratings(train, test)
```

Fit each `LabelEncoder` on the **combined train and test IDs**, transform both
tables, preserve row order, and return integer NumPy arrays. This is the same
contract used by the supplied models.

In [ ]:
def encode_ratings(train, test):
    """Return encoded arrays and vocabulary sizes."""
    user_encoder = LabelEncoder().fit(
        pd.concat([train['user'], test['user']])
    )
    item_encoder = LabelEncoder().fit(
        pd.concat([train['item'], test['item']])
    )

    # TODO: transform both train and test user/item columns.
    train_users = ...
    test_users = ...
    train_items = ...
    test_items = ...

    # TODO: build the arrays expected by the models.
    X_train = ...
    X_test = ...
    y_train = ...

    return X_train, X_test, y_train, len(user_encoder.classes_), len(item_encoder.classes_)


X_train, X_test, y_train, n_users, n_items = encode_ratings(train, test)
print('X_train:', X_train.shape, '| y_train:', y_train.shape, '| X_test:', X_test.shape)
print('Users:', n_users, '| Items:', n_items)

## 3. Baseline scoreboard (7 min)

Complete the helper `fit_and_score`. It should fit a supplied model on
`X_train, y_train`, predict `X_test`, and return local RMSE.

Then use it to score the three baselines. Which one wins on this mini dataset?

In [ ]:
def fit_and_score(model):
    # TODO: fit, predict, and score.
    ...


baselines = {
    'Global mean': GlobalMeanRS(n_users, n_items),
    'User mean': UserMeanRS(n_users, n_items),
    'Item mean': ItemMeanRS(n_users, n_items),
}

rows = []
for name, model in baselines.items():
    rows.append({'method': name, 'RMSE': fit_and_score(model)})

display(pd.DataFrame(rows).sort_values('RMSE').round(3))

## 4. Fit the starting MF model (5 min)

The supplied `BiasedMF` predicts

$$\widehat r_{ui}=\mu+a_u+b_i+p_u^\mathsf{T}q_i.$$

Fit the starting model below, predict `X_test`, and score it. This model is
**not** tuned yet.

In [ ]:
untuned = BiasedMF(
    n_users=n_users, n_items=n_items,
    K=2, lambda_=3.0, n_iters=3, random_state=RANDOM_STATE,
)

# TODO: fit, predict, and compute local RMSE.
...

print('starting MF RMSE:', round(..., 3))

## 5. Tune with 4-fold cross-validation (10 min)

Use `GridSearchCV` with these **rehearsal** settings:

- `K = [2, 3]`
- `lambda_ = [0.5, 2.0]`
- `cv = 4`
- `scoring = 'neg_root_mean_squared_error'`
- `refit=True`

Then display the CV table and use `best_estimator_` for the local test score.

In [ ]:
param_grid = {
    'K': [2, 3],
    'lambda_': [0.5, 2.0],
}

mf_search = GridSearchCV(
    estimator=BiasedMF(
        n_users=n_users, n_items=n_items,
        n_iters=3, random_state=RANDOM_STATE,
    ),
    param_grid=param_grid,
    scoring='neg_root_mean_squared_error',
    cv=4,
    refit=True,
)

# TODO: fit the search, display the CV table, and score the best estimator.
...

cv_table = pd.DataFrame(mf_search.cv_results_)[
    ['param_K', 'param_lambda_', 'mean_test_score', 'rank_test_score']
].copy()
cv_table['validation_RMSE'] = -cv_table.pop('mean_test_score')
display(cv_table.sort_values('validation_RMSE').round(3))

print('selected settings:', ...)
tuned_pred = ...
print('tuned MF RMSE:', round(root_mean_squared_error(test_labels, tuned_pred), 3))

## 6. Compare and explain (3 min)

Fill the comparison table from your outputs:

| Model | Local RMSE |
| --- | ---: |
| Global mean | ... |
| User mean | ... |
| Item mean | ... |
| Starting MF | ... |
| Tuned MF | ... |

1. Which model has the lowest local RMSE?
2. Did tuning improve on the starting MF?
3. Why should model selection use validation folds rather than `test_labels`?
4. What would you check before predicting for a user who never appeared in training?

**Exam checklist:** read the data → encode IDs consistently → preserve order →
baselines → MF with CV → predict `X_test` in order → submit.